# YĀTRĀ AI — Notebook 03: Data Preprocessing & Leakage-Free Partitioning
## Ensuring Enterprise Isolation and Pre-Choice Feature Integrity

This notebook details:
1. Canonical data cleaning and coordinate normalization (`src.data_cleaning`)
2. Pre-training leakage audit of the discrete choice dataset (`src.feature_engineering`)
3. Deterministic traveller-level 70/15/15 train/val/test data partition
4. Scikit-Learn `ColumnTransformer` pipelines for linear and tree-based models


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

from src.feature_engineering import perform_leakage_audit, split_by_traveller, build_feature_matrices
print("Feature engineering modules imported.")


### 1. Loading Frozen Choice Dataset (138,603 Observations)


In [ ]:
choice_path = os.path.join(BASE_DIR, "data", "synthetic", "choice_dataset.parquet")
df = pd.read_parquet(choice_path)
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target Distribution: {dict(df['chosen'].value_counts())}")
print(f"Positive Class (chosen = 1) Rate: {df['chosen'].mean() * 100:.2f}% (Class Imbalance: 2.465 : 1)")


### 2. Strict Pre-Training Leakage Audit
We verify that post-choice variables (`choice_probability`, `utility`, `rank`) are strictly excluded, while normalized candidate quality scores are pre-choice attributes.


In [ ]:
df_audit = perform_leakage_audit(df)
print(f"Columns Audited: {len(df_audit)}")
print(f"Features in Core Model A: {df_audit['in_core_model_a'].sum()}")
print(f"Features in Extended Model B: {df_audit['in_extended_model_b'].sum()}")
df_audit[df_audit['classification_tier'].isin(['TARGET', 'LEAKAGE / POST-CHOICE', 'DERIVED CANDIDATE SCORE'])][['column_name', 'classification_tier', 'in_core_model_a', 'description']]


### 3. Deterministic Traveller-Level Partitioning
To prevent entity leakage, splitting is performed strictly by `traveller_id`:
- **Train**: 70% of travellers (3,500 travellers, 97,163 observations)
- **Validation**: 15% of travellers (750 travellers, 20,715 observations)
- **Test**: 15% of travellers (750 travellers, 20,725 observations)


In [ ]:
df_train, df_val, df_test, df_split = split_by_traveller(df, seed=42)
df_split[['partition', 'traveller_count', 'total_rows', 'positive_rate_pct', 'imbalance_ratio']]


### 4. Zero Leakage Verification
We programmatically assert that no traveller appears in more than one partition.


In [ ]:
train_tids = set(df_train['traveller_id'])
val_tids = set(df_val['traveller_id'])
test_tids = set(df_test['traveller_id'])

assert len(train_tids.intersection(val_tids)) == 0, "Train-Val Leakage!"
assert len(train_tids.intersection(test_tids)) == 0, "Train-Test Leakage!"
assert len(val_tids.intersection(test_tids)) == 0, "Val-Test Leakage!"
print("ZERO ENTITY LEAKAGE CONFIRMED across all three partitions.")


### 5. ColumnTransformer Matrix Construction


In [ ]:
matrices = build_feature_matrices(df_train, df_val, df_test, feature_config="core")
print("Core Linear Matrix Shape (Train):", matrices['linear']['X_train'].shape)
print("Core Tree Matrix Shape (Train):", matrices['tree']['X_train'].shape)
print("Target Vector Shape (Train):", matrices['y_train'].shape)
